# CUSUM Accumulation Timeline Viewer

Use this notebook to inspect how `positive_sum` and `negative_sum` accumulate and reset for one selected CUSUM setting. It does not run the full grid. It only computes and saves the selected timeline.

In [7]:
from pathlib import Path
import json
import subprocess
import sys

from IPython.display import Image, display, Markdown, clear_output

try:
    import ipywidgets as widgets
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
EXPERIMENTS_DIR = PROJECT_ROOT / "results" / "kelmarsh" / "experiments"
DEFAULT_RUN_ID = "all6_multi3_seq12_h64_l1_bs64_lr1e-03_wd0_do0_e60_p8_seed42"
TURBINES = [f"Kelmarsh_{i}" for i in range(1, 7)]
YEARS = [2023, 2024]
MONTHS = [("All year", 0)] + [(f"{m:02d}", m) for m in range(1, 13)]
K_OPTIONS = [round(i / 10, 1) for i in range(10)] + list(range(1, 11))
H_OPTIONS = list(range(20, 201, 20))
BUFFER_OPTIONS = [("Changed buffer: ±6h", 6), ("Old buffer: ±1h", 1)]

def list_experiments():
    if not EXPERIMENTS_DIR.exists():
        return [DEFAULT_RUN_ID]
    runs = sorted(p.name for p in EXPERIMENTS_DIR.iterdir() if (p / "metadata.json").exists())
    return runs or [DEFAULT_RUN_ID]

def load_targets(run_id):
    metadata_path = EXPERIMENTS_DIR / run_id / "metadata.json"
    if not metadata_path.exists():
        return [
            "Generator bearing front temperature (°C)",
            "Stator temperature 1 (°C)",
            "Rear bearing temperature (°C)",
        ]
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    return metadata.get("targets", [])

def run_cusum_timeline(run_id, turbine, target, year, month, k, h, buffer_hours=6, reset_on_alarm=True, hide_non_operation_alarms=False):
    cmd = [
        sys.executable,
        str(PROJECT_ROOT / "scripts" / "plotting" / "main_plot_cusum_timeline.py"),
        "--run-id", run_id,
        "--turbine", turbine,
        "--target", target,
        "--year", str(year),
        "--month", str(month),
        "--k", str(k),
        "--h", str(h),
        "--informational-env-buffer-hours", str(buffer_hours),
    ]
    if not reset_on_alarm:
        cmd.append("--no-reset")
    if hide_non_operation_alarms:
        cmd.append("--hide-non-operation-alarms")
    result = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("CUSUM timeline generation failed")
    fig_path = None
    csv_path = None
    for line in result.stdout.splitlines():
        if line.startswith("Timeline figure saved to:"):
            fig_path = line.split(":", 1)[1].strip()
        if line.startswith("Timeline CSV saved to:"):
            csv_path = line.split(":", 1)[1].strip()
    if csv_path:
        display(Markdown(f"Saved CSV: `{csv_path}`"))
    if fig_path:
        display(Image(filename=fig_path))
    return fig_path, csv_path


In [8]:
if HAS_WIDGETS:
    run_dropdown = widgets.Dropdown(options=list_experiments(), value=DEFAULT_RUN_ID, description="Run")
    turbine_dropdown = widgets.Dropdown(options=TURBINES, value="Kelmarsh_1", description="Turbine")
    target_dropdown = widgets.Dropdown(options=load_targets(DEFAULT_RUN_ID), description="Target")
    year_dropdown = widgets.Dropdown(options=YEARS, value=2023, description="Year")
    month_dropdown = widgets.Dropdown(options=MONTHS, value=1, description="Period")
    k_dropdown = widgets.Dropdown(options=K_OPTIONS, value=0.0, description="k")
    h_dropdown = widgets.Dropdown(options=H_OPTIONS, value=140, description="h")
    buffer_dropdown = widgets.Dropdown(options=BUFFER_OPTIONS, value=6, description="Buffer")
    reset_checkbox = widgets.Checkbox(value=True, description="Reset after alarm")
    hide_nonop_checkbox = widgets.Checkbox(value=False, description="Hide non-operation alarms")
    button = widgets.Button(description="Generate timeline", button_style="primary")
    output = widgets.Output()

    def refresh_targets(change):
        targets = load_targets(change["new"])
        target_dropdown.options = targets
        if targets:
            target_dropdown.value = targets[0]

    def on_click(_):
        with output:
            clear_output(wait=True)
            run_cusum_timeline(
                run_dropdown.value,
                turbine_dropdown.value,
                target_dropdown.value,
                year_dropdown.value,
                month_dropdown.value,
                k_dropdown.value,
                h_dropdown.value,
                buffer_dropdown.value,
                reset_checkbox.value,
                hide_nonop_checkbox.value,
            )

    run_dropdown.observe(refresh_targets, names="value")
    button.on_click(on_click)
    controls = widgets.VBox([
        widgets.HBox([run_dropdown]),
        widgets.HBox([turbine_dropdown, target_dropdown]),
        widgets.HBox([year_dropdown, month_dropdown, k_dropdown, h_dropdown]),
        widgets.HBox([buffer_dropdown, reset_checkbox, hide_nonop_checkbox]),
        button,
        output,
    ])
    display(controls)
else:
    print("ipywidgets is not available. Use run_cusum_timeline(...) manually.")


In [9]:
# Manual example if widgets are unavailable:
# run_cusum_timeline(
#     DEFAULT_RUN_ID,
#     turbine="Kelmarsh_1",
#     target="Rear bearing temperature (°C)",
#     year=2023,
#     month=1,
#     k=0.0,
#     h=140,
#     buffer_hours=6,
#     reset_on_alarm=True,
#     hide_non_operation_alarms=False,
# )
